# Import dependencies

In [2]:
pip install -r "requirements.txt"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np

# Data

## Load resale data

In [175]:
resale = pd.read_csv("Data/Resale_with_Coords.csv")

In [176]:
print(f"Shape: {resale.shape}")
print(resale.info())
resale.describe(include='all').T

Shape: (108, 14)
<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                108 non-null    str    
 1   town                 108 non-null    str    
 2   flat_type            108 non-null    str    
 3   block                108 non-null    str    
 4   street_name          108 non-null    str    
 5   storey_range         108 non-null    str    
 6   floor_area_sqm       108 non-null    int64  
 7   flat_model           108 non-null    str    
 8   lease_commence_date  108 non-null    int64  
 9   remaining_lease      108 non-null    str    
 10  resale_price         108 non-null    int64  
 11  address              108 non-null    str    
 12  latitude             108 non-null    float64
 13  longitude            108 non-null    float64
dtypes: float64(2), int64(3), str(9)
memory usage: 11.9 KB
None


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
month,108,1,2017-01,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
town,108,2,ANG MO KIO,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flat_type,108,5,3 ROOM,63,NaN,NaN,NaN,NaN,NaN,NaN,NaN
block,108,92,709,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street_name,108,22,ANG MO KIO AVE 10,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
storey_range,108,7,04 TO 06,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area_sqm,108.0,NaN,NaN,NaN,79.12037,17.035267,44.0,67.0,73.0,91.0,147.0
flat_model,108,7,New Generation,72,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lease_commence_date,108.0,NaN,NaN,NaN,1981.472222,7.363028,1974.0,1978.0,1980.0,1981.0,2012.0
remaining_lease,108,55,62 years 05 months,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Data Check [None]

In [177]:
pd.DataFrame({
    'nunique': resale.nunique(dropna=False),
    'missing': resale.isna().sum(),
}).sort_values('nunique', ascending=False)

,nunique,missing
longitude,97,0
latitude,97,0
address,97,0
block,92,0
resale_price,70,0
remaining_lease,55,0
floor_area_sqm,31,0
street_name,22,0
lease_commence_date,17,0
storey_range,7,0


In [178]:
resale=resale.dropna()

## Data Description

In [179]:
# 'role': 'feature', 'target', 'identifier', 'metadata', 'ambiguous'
# 'type':  'num-cont', 'num-disc', 'cat-nom', 'cat-ord', 'bool'
role_map = {
    "month" : "feature",
    "town": "feature",
    "flat_type": "feature",
    "block": "identifier",
    "street_name": "identifier",
    "storey_range": "feature",
    "floor_area_sqm": "feature",
    "flat_model": "metadata",
    "lease_commence_date": "metadata",
    "remaining_lease": "feature",
    "resale_price": "target"
}
type_map = {
    "month" : "num-disc",
    "town": "cat-nom",
    "flat_type": "cat-nom",
    "block": "cat-nom",
    "street_name": "cat-nom",
    "storey_range": "cat-ord",
    "floor_area_sqm": "num-disc",
    "flat_model": "cat-nom",
    "lease_commence_date": "num-disc",
    "remaining_lease": "num-disc",
    "resale_price": "num-disc"
}

data_description = pd.DataFrame({
    "column": resale.columns,
})
data_description["role"] = data_description["column"].map(role_map).fillna("unknown")
data_description["type"] = data_description["column"].map(type_map).fillna("unknown")
data_description

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,feature,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
7,flat_model,metadata,cat-nom
8,lease_commence_date,metadata,num-disc
9,remaining_lease,feature,num-disc


## Data Pre-Processing

### Model Dataset Descriptions

In [180]:
model_data_desc = data_description[data_description['role'] != "metadata"]
model_data_desc

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,feature,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
9,remaining_lease,feature,num-disc
10,resale_price,target,num-disc
11,address,unknown,unknown


### Model Dataset

In [181]:
model_data = resale[model_data_desc["column"]].copy()
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,61 years 04 months,232000,406 ANG MO KIO AVE 10,1.362005,103.853880
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,60 years 07 months,250000,108 ANG MO KIO AVE 4,1.370966,103.838202
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,262000,602 ANG MO KIO AVE 5,1.380709,103.835368
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,62 years 01 month,265000,465 ANG MO KIO AVE 10,1.366201,103.857201
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,265000,601 ANG MO KIO AVE 5,1.381041,103.835132


### Data Transformation

**Month** \
Convert to indexing for model input, index represents the order for time-series models \
index starts from 2017-01 onwards \
Example:

| Month(before) | index(after) |
| ------ | ------ |
| 2017-01 | 0 |
| 2017-02 | 1 |

In [182]:
months = pd.to_datetime(model_data['month'], format='%Y-%m')
model_data['month'] = (
    months.dt.year * 12 + months.dt.month
)

model_data['month'] -= model_data['month'].min()
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,61 years 04 months,232000,406 ANG MO KIO AVE 10,1.362005,103.853880
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,60 years 07 months,250000,108 ANG MO KIO AVE 4,1.370966,103.838202
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,262000,602 ANG MO KIO AVE 5,1.380709,103.835368
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,62 years 01 month,265000,465 ANG MO KIO AVE 10,1.366201,103.857201
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,265000,601 ANG MO KIO AVE 5,1.381041,103.835132


**remaining_lease [Months]** \
Convert remaining_lease to number of months instead of X years X months

In [183]:
remaining_yrs = model_data['remaining_lease'].str.extract(r'(\d+)\D+(?:(\d+)\D+)?').fillna(0).astype(int)
model_data['remaining_lease'] = remaining_yrs[0] * 12 + remaining_yrs[1]
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132


**Storey type** \
Replace storey range with the storey type (lower, middle, upper)

In [184]:
avg_storey = model_data['storey_range'].str.split(" TO ", expand=True).astype(int).mean(axis=1)
model_data['storey_type'] = pd.cut(avg_storey, bins=[1,3, 7,99], labels=['lower','middle','upper'], right=False)
model_data = pd.get_dummies(model_data, columns=['storey_type'])
model_data.drop(columns=['storey_range'], inplace=True)
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False


**Quarter index** \
Defines which quarter is the data in example 2 for 2017-04 to 2017-07

In [185]:
model_data['quarter'] = model_data['month'] // 3
model_data.tail()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter
103,0,BEDOK,4 ROOM,554,BEDOK NTH ST 3,93,749,412000,554 BEDOK NTH ST 3,1.332627,103.928172,False,True,False,0
104,0,BEDOK,4 ROOM,602,BEDOK RESERVOIR RD,98,772,420000,602 BEDOK RESERVOIR RD,1.329349,103.911583,False,False,True,0
105,0,BEDOK,4 ROOM,81,BEDOK NTH RD,91,725,425000,81 BEDOK NTH RD,1.328916,103.940529,False,False,True,0
106,0,BEDOK,4 ROOM,508,BEDOK NTH AVE 3,92,730,430000,508 BEDOK NTH AVE 3,1.333400,103.932519,False,False,True,0
107,0,BEDOK,4 ROOM,720,BEDOK RESERVOIR RD,104,793,430000,720 BEDOK RESERVOIR RD,1.335973,103.924883,False,True,False,0


**Resale Price Index [RPI]**

In [186]:
rpi = pd.read_csv('Data/2025-RPI.csv')
rpi = rpi[rpi['year'] >= 2017]
rpi['quarter'] = (rpi['year'] - 2017) * 4 + (rpi['quarter'] - 1)
rpi.drop(columns=['year'], inplace=True)
rpi.head()

,quarter,rpi
32,0,133.9
33,1,133.7
34,2,132.8
35,3,132.6
36,4,131.6


In [187]:
model_data = pd.merge(model_data, rpi, on='quarter',how='left')
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter,rpi
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True,0,133.9
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False,0,133.9
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False,0,133.9
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False,0,133.9
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False,0,133.9


**Adjusted Resale Price** \
Formula: Resale price / RPI

In [188]:
model_data['adjusted_resale'] = model_data['resale_price'] / (model_data['rpi'] / 100)
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter,rpi,adjusted_resale
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True,0,133.9,173263.629574
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False,0,133.9,186706.497386
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False,0,133.9,195668.409261
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False,0,133.9,197908.887229
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False,0,133.9,197908.887229


**Remove unneccessary columns**

In [189]:
transformed = model_data.drop(columns=['block','street_name','quarter','address'])
transformed.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,True,133.9,173263.629574
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,False,133.9,186706.497386
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,False,133.9,195668.409261
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,False,133.9,197908.887229
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,False,133.9,197908.887229


# Feature Extraction

Add amenity count for the varies amenities (mrt,bus-stop, schools, etc..)

## Import Dependencies

In [190]:
import geopandas as gpd
from shapely.geometry import Point

## Load Data

In [191]:
mrt_df = pd.read_csv("Data/mrt_stations.csv")
schools_df = pd.read_csv("Data/schools.csv")
malls_df = pd.read_csv("Data/shopping_malls.csv")
bus_stop_df = pd.read_csv("Data/bus_stops.csv")
hawker_df = pd.read_csv("Data/hawker_centres.csv")
wet_market_df = pd.read_csv("Data/wet_markets.csv")

## Project Data on Point

In [192]:
mrt_gdf = gpd.GeoDataFrame(mrt_df, geometry=gpd.points_from_xy(mrt_df.long, mrt_df.lat), crs="EPSG:4326")
schools_gdf = gpd.GeoDataFrame(schools_df, geometry=gpd.points_from_xy(schools_df.long, schools_df.lat), crs="EPSG:4326")
malls_gdf = gpd.GeoDataFrame(malls_df, geometry=gpd.points_from_xy(malls_df.long, malls_df.lat), crs="EPSG:4326")
bus_stop_gdf = gpd.GeoDataFrame(bus_stop_df, geometry=gpd.points_from_xy(bus_stop_df.long, bus_stop_df.lat), crs="EPSG:4326")
hawker_gdf = gpd.GeoDataFrame(hawker_df, geometry=gpd.points_from_xy(hawker_df.long, hawker_df.lat), crs="EPSG:4326")
wet_market_gdf = gpd.GeoDataFrame(hawker_df, geometry=gpd.points_from_xy(hawker_df.long, hawker_df.lat), crs="EPSG:4326")

In [193]:
flat_gdf = gpd.GeoDataFrame(transformed, geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude), crs="EPSG:4326")
flat_gdf.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale,geometry
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,True,133.9,173263.629574,POINT (103.85388 1.362)
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,False,133.9,186706.497386,POINT (103.8382 1.37097)
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,False,133.9,195668.409261,POINT (103.83537 1.38071)
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,False,133.9,197908.887229,POINT (103.8572 1.3662)
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,False,133.9,197908.887229,POINT (103.83513 1.38104)


In [194]:
# Reproject to a Metric CRS (Singapore SVY21 is EPSG:3414)
flat_gdf = flat_gdf.to_crs(epsg=3414)
mrt_gdf = mrt_gdf.to_crs(epsg=3414)
schools_gdf = schools_gdf.to_crs(epsg=3414)
malls_gdf = malls_gdf.to_crs(epsg=3414)
bus_stop_gdf = bus_stop_gdf.to_crs(epsg=3414)
hawker_gdf = hawker_gdf.to_crs(epsg=3414)
wet_market_gdf = wet_market_gdf.to_crs(epsg=3414)

## Create Buffer zone

In [195]:
# Create a 500m buffer around each house
flat_gdf['geometry'] = flat_gdf.geometry.buffer(500)

## Get Nearby amenties in radius (500M) (New Feature columns)

In [196]:
# Spatial Join: Count MRT stations inside the house buffers
joined = gpd.sjoin(flat_gdf, mrt_gdf, how="left", predicate="intersects")
school_joined = gpd.sjoin(flat_gdf, schools_gdf, how="left", predicate="intersects")
mall_joined = gpd.sjoin(flat_gdf, malls_gdf, how="left", predicate="intersects")
bus_stop_joined = gpd.sjoin(flat_gdf, bus_stop_gdf, how="left", predicate="intersects")
hawker_joined = gpd.sjoin(flat_gdf, hawker_gdf, how="left", predicate="intersects")
wet_market_joined = gpd.sjoin(flat_gdf, wet_market_gdf, how="left", predicate="intersects")

mrt_counts = joined.groupby(joined.index).size() - joined['index_right'].isna().groupby(joined.index).sum()
school_counts = school_joined.groupby(school_joined.index).size() - school_joined['index_right'].isna().groupby(school_joined.index).sum()
mall_counts = mall_joined.groupby(mall_joined.index).size() - mall_joined['index_right'].isna().groupby(mall_joined.index).sum()
bus_stop_counts = bus_stop_joined.groupby(bus_stop_joined.index).size() - bus_stop_joined['index_right'].isna().groupby(bus_stop_joined.index).sum()
hawker_counts = hawker_joined.groupby(hawker_joined.index).size() - hawker_joined['index_right'].isna().groupby(hawker_joined.index).sum()
wet_market_counts = wet_market_joined.groupby(wet_market_joined.index).size() - wet_market_joined['index_right'].isna().groupby(wet_market_joined.index).sum()

transformed['mrt_count'] = mrt_counts.values
transformed['school_count'] = school_counts.values
transformed['mall_count'] = mall_counts.values
transformed['bus_stop_count'] = bus_stop_counts.values
transformed['hawker_count'] = hawker_counts.values
transformed['wet_market_count'] = wet_market_counts.values
transformed.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,True,133.9,173263.629574,0,3,0,14,1,1
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,False,133.9,186706.497386,1,2,0,11,1,1
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,False,133.9,195668.409261,1,0,0,12,0,0
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,False,133.9,197908.887229,0,1,0,10,2,2
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,False,133.9,197908.887229,1,0,0,11,0,0


## Calculate distance to nearest bus-stop and MRT station in meters (New Feature Columns)

In [197]:
# Create GeoDataFrame from transformed data
flat_point_gdf = gpd.GeoDataFrame(
    transformed, 
    geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude), 
    crs="EPSG:4326"
)
flat_point_gdf = flat_point_gdf.to_crs(epsg=3414)
print("✓ flat_point_gdf created and reprojected.")

# --- Calculate Distance to Nearest MRT ---
print("Calculating nearest MRT distances...")
mrt_distances = []

for idx, flat in flat_point_gdf.iterrows():
    distances_to_all_mrt = mrt_gdf.geometry.distance(flat.geometry)
    min_distance = distances_to_all_mrt.min()
    mrt_distances.append(min_distance)

transformed['nearest_mrt_dist'] = pd.Series(mrt_distances).round(0).astype(int)
print(f"✓ Nearest MRT distance calculated for {len(mrt_distances)} flats.")

# --- Calculate Distance to Nearest Bus Stop ---
print("Calculating nearest bus stop distances...")
bus_distances = []

for idx, flat in flat_point_gdf.iterrows():
    distances_to_all_bus = bus_stop_gdf.geometry.distance(flat.geometry)
    min_distance = distances_to_all_bus.min()
    bus_distances.append(min_distance)

transformed['nearest_bus_stop_dist'] = pd.Series(bus_distances).round(0).astype(int)
print(f"✓ Nearest bus stop distance calculated for {len(bus_distances)} flats.")

# --- Validation ---
print(f"\nValidation:")
print(f"  MRT - NaN values: {transformed['nearest_mrt_dist'].isna().sum()}")
print(f"  Bus - NaN values: {transformed['nearest_bus_stop_dist'].isna().sum()}")
print(f"  MRT - Mean distance: {transformed['nearest_mrt_dist'].mean():.0f}m")
print(f"  Bus - Mean distance: {transformed['nearest_bus_stop_dist'].mean():.0f}m")

display(transformed.head())

✓ flat_point_gdf created and reprojected.
Calculating nearest MRT distances...
✓ Nearest MRT distance calculated for 108 flats.
Calculating nearest bus stop distances...
✓ Nearest bus stop distance calculated for 108 flats.

Validation:
  MRT - NaN values: 0
  Bus - NaN values: 0
  MRT - Mean distance: 574m
  Bus - Mean distance: 123m


,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,...,133.9,173263.629574,0,3,0,14,1,1,1016,91
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,...,133.9,186706.497386,1,2,0,11,1,1,202,164
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,...,133.9,195668.409261,1,0,0,12,0,0,460,136
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,...,133.9,197908.887229,0,1,0,10,2,2,828,68
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,...,133.9,197908.887229,1,0,0,11,0,0,433,146


In [198]:
import numpy as np

transformed['nearest_mrt_dist'] = np.floor(transformed['nearest_mrt_dist'])
transformed['nearest_bus_stop_dist'] = np.floor(transformed['nearest_bus_stop_dist'])

display(transformed.head())

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,...,133.9,173263.629574,0,3,0,14,1,1,1016,91
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,...,133.9,186706.497386,1,2,0,11,1,1,202,164
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,...,133.9,195668.409261,1,0,0,12,0,0,460,136
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,...,133.9,197908.887229,0,1,0,10,2,2,828,68
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,...,133.9,197908.887229,1,0,0,11,0,0,433,146


The `nearest_mrt_dist` and `nearest_bus_stop_dist` columns have now been rounded down to the nearest whole number. You can see the updated values in the displayed DataFrame.

# Export Data

In [200]:
transformed.to_csv('Data/processed.csv', index=False)

# Price Prediction Models

## Load data

In [207]:
import pandas as pd
import numpy as np

resale = pd.read_csv("Data/processed.csv")
resale.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,...,133.9,173263.629574,0,3,0,14,1,1,1016,91
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,...,133.9,186706.497386,1,2,0,11,1,1,202,164
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,...,133.9,195668.409261,1,0,0,12,0,0,460,136
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,...,133.9,197908.887229,0,1,0,10,2,2,828,68
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,...,133.9,197908.887229,1,0,0,11,0,0,433,146


In [208]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Separate features and target
resale = resale.dropna(subset=["nearest_mrt_dist", "nearest_bus_stop_dist"])
X = resale.drop(columns=['adjusted_resale','resale_price', 'latitude', 'longitude', 'month', 'rpi'])
y = resale["adjusted_resale"]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)
print(X.columns)
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Index(['floor_area_sqm', 'remaining_lease', 'storey_type_lower',
       'storey_type_middle', 'storey_type_upper', 'mrt_count', 'school_count',
       'mall_count', 'bus_stop_count', 'hawker_count', 'wet_market_count',
       'nearest_mrt_dist', 'nearest_bus_stop_dist', 'town_BEDOK',
       'flat_type_3 ROOM', 'flat_type_4 ROOM', 'flat_type_5 ROOM',
       'flat_type_EXECUTIVE'],
      dtype='str')


## Dummy Regressor

In [209]:
# Initialize dummy model (predicts mean of y_train)
dummy_model = DummyRegressor(strategy="mean")

# Train
dummy_model.fit(X_train, y_train)

# Predict on test set
dummy_pred = dummy_model.predict(X_test)

# Evaluate
print("Dummy Baseline Performance:")
print("MAE:", mean_absolute_error(y_test, dummy_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dummy_pred)))
print("R2:", r2_score(y_test, dummy_pred))

Dummy Baseline Performance:
MAE: 62801.22271045731
RMSE: 69933.4350754017
R2: -0.5583645860095456


## Linear Regression

In [210]:
# Initialize model
lr_model = LinearRegression()

# Train
lr_model.fit(X_train, y_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
print("Linear Regression Performance:")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("R2:", r2_score(y_test, lr_pred))

Linear Regression Performance:
MAE: 15491.02732034066
RMSE: 23740.076996530148
R2: 0.8204176730442378


## K-Means

In [212]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Default KMeans, set random_state to 1
kmeans =KMeans(random_state=1)

# Fit training data
kmeans.fit(X_train_scaled)

# Predict cluster labels
train_clusters = kmeans.predict(X_train_scaled)
test_clusters = kmeans.predict(X_test_scaled)

# Compute cluster mean prices
cluster_means = {}

for c in range(kmeans.n_clusters):
    cluster_mean = y_train[train_clusters == c].mean()
    cluster_means[c] = cluster_mean

# Predict test prices using cluster mean
kmeans_pred = [cluster_means[c] for c in test_clusters]

print("KMeans Performance:")
print("MAE:", mean_absolute_error(y_test, kmeans_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, kmeans_pred)))
print("R2:", r2_score(y_test, kmeans_pred))
X_test.head()

KMeans Performance:
MAE: 24831.319015396704
RMSE: 36893.49135851402
R2: 0.5662905747234777


,floor_area_sqm,remaining_lease,storey_type_lower,storey_type_middle,storey_type_upper,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist,town_BEDOK,flat_type_3 ROOM,flat_type_4 ROOM,flat_type_5 ROOM,flat_type_EXECUTIVE
77,68,763,False,False,True,1,2,0,10,1,1,458,214,True,True,False,False,False
10,68,745,True,False,False,0,3,0,17,1,1,679,89,False,True,False,False,False
4,67,749,True,False,False,1,0,0,11,0,0,433,146,False,True,False,False,False
83,67,732,False,False,True,0,1,1,21,1,1,611,124,True,True,False,False,False
62,68,760,False,True,False,0,0,0,10,0,0,646,181,True,True,False,False,False


## Random Forest

In [213]:
from sklearn.ensemble import RandomForestRegressor

# Init the RF model
rf_model = RandomForestRegressor(random_state=1)

# Train
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
print("Random Forest Performance:")
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("R2:", r2_score(y_test, rf_pred))

Random Forest Performance:
MAE: 16834.345479762935
RMSE: 25287.520880453972
R2: 0.7962433193183183
